In [ ]:
# Двумерный случай: контурный график и релаксационная последовательность по r

α = 1.0
β = 3.0   # можно менять для экспериментов

rs = [1.0, 10.0, 100.0, 200.0]
colors = [:blue, :green, :orange, :red]

x0 = [-1.0, -1.0]   # стартовая точка (вне допустимой области)

mins = Vector{Vector{Float64}}(undef, length(rs))
paths = Vector{Vector{Vector{Float64}}}(undef, length(rs))

for (i, r) in enumerate(rs)
    xmin, path = gradient_descent_penalty(x0, r; α=α, β=β)
    mins[i] = xmin
    paths[i] = path
end

println("Последовательность найденных минимумов Q(x, r):")
for (i, r) in enumerate(rs)
    println("r = $(r): x*(r) = $(mins[i]), f(x*(r)) = ", f(mins[i], α, β), ", g(x*(r)) = ", g(mins[i]))
end

# Сетка для построения уровней функции f(x)
x1_range = range(-3.0, 4.0, length=200)
x2_range = range(-3.0, 4.0, length=200)
Z = [f([x1, x2], α, β) for x2 in x2_range, x1 in x1_range]

p2d = contour(x1_range, x2_range, Z,
              levels=30,
              xlabel="x₁", ylabel="x₂",
              title="Метод внешних штрафов: контуры f(x) и точки минимума Q(x,r)",
              c=:plasma, colorbar=true)

# Линия ограничения g(x) = 0  =>  x₁ + x₂ = 1
plot!(p2d, x1_range, 1 .- x1_range, color=:black, linewidth=2, label="g(x) = 0")

# Область g(x) ≤ 0 (x₁ + x₂ ≥ 1) заштрихована примерно
fill_x = vcat(collect(x1_range), reverse(collect(x1_range)))
fill_y = vcat(1 .- x1_range, fill(maximum(x2_range), length(x1_range)))
plot!(p2d, fill_x, fill_y, fillrange=minimum(x2_range), alpha=0.1, color=:gray, label="g(x) ≤ 0")

# Релаксационная последовательность минимумов при росте r
for (i, r) in enumerate(rs)
    x_star = mins[i]
    scatter!(p2d, [x_star[1]], [x_star[2]],
             markersize=6,
             color=colors[i],
             marker=:circle,
             label="r = $(Int(r))")
end

scatter!(p2d, [x0[1]], [x0[2]], markersize=8, color=:white, marker=:star5,
         label="Стартовая точка")

p2d


In [ ]:
# Трёхмерный случай: поверхности Q(x,r) (параболоиды) и точки минимума

x1_range3 = range(-3.0, 4.0, length=80)
x2_range3 = range(-3.0, 4.0, length=80)

# Базовая поверхность f(x) для ориентира
Z_f = [f([x1, x2], α, β) for x2 in x2_range3, x1 in x1_range3]

p3d = surface(x1_range3, x2_range3, Z_f,
              xlabel="x₁", ylabel="x₂", zlabel="значение",
              title="Метод внешних штрафов: поверхности Q(x,r)",
              c=:plasma, alpha=0.5, label="f(x)")

# Дополнительные поверхности Q(x,r) для разных r (семейство параболоидов)
for (i, r) in enumerate(rs)
    Z_Q = [Q([x1, x2], r; α=α, β=β) for x2 in x2_range3, x1 in x1_range3]
    surface!(p3d, x1_range3, x2_range3, Z_Q,
             c=colors[i], alpha=0.25, label="Q(x,r=$(Int(r)))")
end

for (i, r) in enumerate(rs)
    x_star = mins[i]
    scatter!(p3d, [x_star[1]], [x_star[2]], [Q(x_star, r; α=α, β=β)],
             markersize=6,
             color=colors[i],
             marker=:sphere,
             label="мин. при r = $(Int(r))")
end

# Линия ограничения g(x)=0 на поверхности f(x)
x1_line = range(-1.0, 3.0, length=50)
x2_line = 1 .- x1_line
z_line = [f([x1, x2], α, β) for (x1, x2) in zip(x1_line, x2_line)]
plot!(p3d, x1_line, x2_line, z_line, color=:black, linewidth=3, label="g(x) = 0")

p3d
